<br>

<img src="https://lindas.admin.ch/static-assets/img/lindaslogo.png" style="width:25%; float:right">

# Tutorial Staatskalender

Dieses Notebook dient dazu, den Staatskalender in RDF/Linked Data besser zu verstehen, um die Daten kompetent nutzen zu können. Es ist **keine Einleitung** - weder in Linked Data noch in SPARQL. Wer die technischen Informationen und Voraussetzungen zu Beginn überspringen möchte, kann direkt zum eigentlichen [Tutorial](#Tutorial) springen.


# Introduction

The webpage you are currently viewing is a so called **interactive JupyterLite notebook**. In this notebook, you can change interactively the content of the single cells and execute these cells directly seeing the result of your changes immediately. The cells contain either [Markdown](https://en.wikipedia.org/wiki/Markdown) content (like this cell) or executable Python source code.

JupyterLite stems from JupyterLab with the advantage of being completely browser based without any backend infrastructure. This means that the execution of the cells could take some time during first execution. Subsequent executions will be much faster because of stored data in your browser cache.

If you are unfamiliar with the handling of Jupyter notebooks, here are two useful resources:

- [The JupyterLab Interface](https://jupyterlab.readthedocs.io/en/stable/user/interface.html)
- [The Jupyter Notebook](https://jupyterlab.readthedocs.io/en/stable/user/notebook.html)

# Preliminaries

In this part some preliminaries for querying LINDAS with SPARQL are introduced. If you are only interested in the actual LINDAS tutorial, you can skip the whole section and start [here](#Actual-Tutorial).

## Install and Import the Necessary Modules

Querying a SPARQL endpoint is basically making a POST request to the corresponding endpoint URL. The necessary code is in the [SPARQL.py](sparql.py) file and is imported in the following cell together with all the necessary modules to work with the data:

In [ ]:
import pandas as pd
import numpy as np
from sparql import query, display_result

# Tutorial

Der Staatskalender ist eine Art "Organigramm" der Bundesbehörden. Da dieses Organigramm einer gewissen Dynamik unterworfen ist (Behörden ändern ihren Namen oder werden mit anderen Behörden zusammengelegt, etc.), ist der Staatskalender in RDF/Linked Data gemäss dem unter https://version.link beschriebenen Datenmodell erstellt. Dieses Datenmodell soll im nachfolgenden Teil erkundet werden.

## Behörden und ihr jeweiliger Name

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX vl: <https://version.link/>

SELECT * WHERE {

    ?org a vl:Identity;
        vl:inVersionedIdentitySet <https://ld.admin.ch/ou>;
        schema:name ?name.
        
    FILTER(lang(?name) = 'de')
}

LIMIT 10

""")

display_result(df)

## Anzahl Behörden

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX vl: <https://version.link/>

SELECT (COUNT(?org) as ?number) WHERE {

    ?org a vl:Identity;
        vl:inVersionedIdentitySet <https://ld.admin.ch/ou>;
}

""")

display_result(df)

## Behörden, die so nicht mehr existieren

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX vl: <https://version.link/>

SELECT * WHERE {

    ?org a vl:Identity;
        vl:inVersionedIdentitySet <https://ld.admin.ch/ou>;
        schema:name ?name;
        a vl:Deprecated.
        
    FILTER(lang(?name) = 'de')
} 

LIMIT 10

""")

display_result(df)

## Behörden mit mehreren Versionen

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX vl: <https://version.link/>

SELECT ?id ?name ?versions WHERE {
    
    ?id schema:name ?name.
    
    FILTER(lang(?name) = 'de')
    FILTER(?number > 5)
        
    {
    SELECT ?id (GROUP_CONCAT(?version; separator=" ← ") AS ?versions) (COUNT(?version) AS ?number) WHERE {

      ?version a vl:Version;
          vl:identity ?id.

    } 
    GROUP BY ?id
    }
}

ORDER BY DESC(?number)

""")

display_result(df)

## "Wiederauferstandene" Behörden

Die nachfolgenden Versionen haben vl:successor mit schema:startDate > als schema:endDate der Version, es existiert also eine zeitliche Lücke, in der die Behörde "pausiert" hat.

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX vl: <https://version.link/>

SELECT * WHERE {

?version vl:successor ?succ;
    vl:inVersionedIdentitySet <https://ld.admin.ch/ou>;
    schema:endDate ?endDate.

?succ schema:startDate ?startDate.

BIND(day(?startDate - ?endDate) as ?diff)
FILTER(?diff > 0)

}
ORDER BY DESC(?diff)

""")

display_result(df)

## Behörden mit gleichen Namen

Dieses Beispiel zeigt, wie die von der SPARQL Query gelieferte Daten mit dem gesamten Spektrum der Datenverarbeitungsmöglichkeit von Python weiterverarbeitet werden kann.

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX vl: <https://version.link/>

SELECT * WHERE {

?identity a vl:Identity;
    schema:name ?name.
    
FILTER(lang(?name) = 'de')

}

ORDER BY ?name

""")

df['non_unique'] = df.duplicated(subset = 'name', keep = False)

df = df[df['non_unique'] == True]

display_result(df[["identity", "name"]])

## Behörden und ihr Pfad im Organigramm

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX vl: <https://version.link/>

SELECT (GROUP_CONCAT(CONCAT(STR(?level), ":", STR(?parent_name)); separator= " → ") AS ?hierarchy) WHERE {

    BIND ('de' AS ?lang)

    ?parent schema:subOrganization* ?org.
    ?parent schema:name ?parent_name.
    FILTER(lang(?parent_name) = ?lang)

    {
    SELECT ?parent (COUNT(?org) AS ?level) WHERE {

        BIND (<https://ld.admin.ch/ou/org-units> AS ?root) #start root, e.g. <https://ld.admin.ch/ou/10000000> = Bundesrat

        ?root schema:subOrganization* ?parent.
        ?org schema:subOrganization* ?parent.


    } GROUP BY ?parent ORDER BY ?level
    }
} GROUP BY ?org ORDER BY ?hierarchy

""")

display_result(df)

## Behörden, die neu gegründet wurden

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX vl: <https://version.link/>

SELECT ?org ?name ?date WHERE {

    ?org a vl:Identity;
        vl:version ?version;
        schema:name ?name.
        
    FILTER(lang(?name) = "de")
        
    OPTIONAL {?version vl:predecessor ?predecessor.}
    ?version schema:startDate ?date.
    
    FILTER(!BOUND(?predecessor))
    FILTER(?date > "2023-01-01")

}

""")

display_result(df)